# Checkout Delivery Analysis

## Контекст исследования

Product рассматривает возможность расширения вариантов получения заказа в Web Checkout за счёт внешней сети partner pickup points.

Product Analytics и Customer Support зафиксировали сигналы о том, что часть пользователей может прекращать Checkout на Delivery step из-за того, что доступные варианты получения заказа не соответствуют их потребностям.

Связь между доступностью delivery options и Checkout drop-off пока не установлена и должна быть исследована до определения scope инициативы и формирования solution requirements.

## Аналитический вопрос

Что имеющиеся данные Product Analytics показывают о переходе пользователей с Delivery на Payment и наблюдается ли связь между доступными delivery options и drop-off на Delivery step?

## Scope анализа

- Канал: Web Checkout
- Funnel transition: Delivery → Payment
- Период: 2026-08-17 — 2026-09-13
- География: Kharkiv, Kyiv, Dnipro, Odesa
- Единица анализа: Checkout session

## Границы интерпретации

Анализ может выявить поведенческие patterns и статистические связи между характеристиками Checkout sessions.

Он не позволяет установить причину, по которой конкретный пользователь прекратил Checkout, и не может доказать, что внедрение partner pickup points приведёт к снижению drop-off.

## Dataset Contract

Dataset предоставлен Product Analytics в виде session-level extract за четыре полные недели.

### Population

Web Checkout sessions, достигшие Delivery step в течение анализируемого периода.

Internal/test traffic и sessions, затронутые известным incident в Delivery service, были исключены Product Analytics до формирования extract.

### Grain

Одна строка представляет одну уникальную Checkout session.

### Поля

| Field | Значение |
|---|---|
| `checkout_session_id` | Уникальный идентификатор Checkout session |
| `session_date` | Дата Checkout session |
| `city` | Город назначения доставки; для Pickup — город, выбранный на Delivery step |
| `delivery_options` | Delivery methods, доступные после первого успешного availability calculation |
| `delivery_step_reached` | Достигла ли session Delivery step |
| `selected_delivery_method` | Последний delivery method, выбранный в session; значение может отсутствовать |
| `payment_step_reached` | Перешла ли session на Payment; не означает успешную оплату |
| `order_completed` | Завершилась ли Checkout session созданием заказа |

### Известные ограничения

- Dataset не содержит причины, по которой пользователь прекратил Checkout.
- Cohorts с различными `delivery_options` являются observational groups, а не эквивалентными experimental groups.
- Доступность delivery methods зависит от location и потенциально от других условий.
- `delivery_options` отражает первый успешный availability calculation и не сохраняет последующие изменения доступности в рамках session.
- `selected_delivery_method` отражает последний выбранный method и поэтому может относиться к более позднему состоянию той же session.



## Подготовка анализа

Для работы с session-level extract используются:

- `Path` — для явного определения расположения входного dataset;
- `pandas` — для загрузки, проверки, преобразования и анализа табличных данных.

На этом этапе библиотеки только подключаются; никаких изменений исходных данных не выполняется.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

### Проверка доступности входного dataset

Перед загрузкой проверяется наличие файла по ожидаемому пути.

Это позволяет отделить проблему доступности input data от возможных ошибок чтения или качества самого dataset.

In [2]:
DATA_PATH = Path(
    r"C:\My-Projects\portfolio-analysis-data\case-01"
    r"\case1_checkout_sessions_2026-08-17_2026-09-13.csv"
)

DATA_PATH.exists()

True

### Загрузка dataset

CSV загружается в `pandas DataFrame`.

Сразу после загрузки проверяется размер dataset — количество observations и fields — для сопоставления с информацией, предоставленной Product Analytics.

In [3]:
df = pd.read_csv(DATA_PATH)

df.shape

(80000, 8)

Dataset содержит 80 000 observations и 8 fields, что соответствует заявленному размеру session-level extract.

### Первичная визуальная инспекция

Просматриваются первые observations dataset для первоначальной проверки структуры, названий fields и общего представления значений.

Эта проверка является ознакомительной и сама по себе не подтверждает качество dataset в целом.

In [4]:
df.head()

,checkout_session_id,session_date,city,delivery_options,delivery_step_reached,selected_delivery_method,payment_step_reached,order_completed
0,CHK-00002870,2026-08-31,Dnipro,courier_only,True,courier,True,True
1,CHK-00047422,2026-08-18,Kyiv,courier_only,True,courier,True,True
2,CHK-00029385,2026-08-22,Odesa,courier_only,True,courier,True,True
3,CHK-00024269,2026-09-12,Kharkiv,courier_and_pickup,True,pickup,True,True
4,CHK-00017617,2026-08-21,Odesa,courier_and_pickup,True,courier,True,False


## Data Quality Assessment

Перед расчётом business metrics dataset проверяется на соответствие заявленному Dataset Contract.

Проверки охватывают:

- размер и schema dataset;
- uniqueness `checkout_session_id`;
- data types;
- missing values;
- допустимые domain values;
- logical consistency между связанными полями.

Цель этапа — определить, достаточно ли надёжен dataset для последующего анализа Delivery → Payment funnel.

### Schema и data types

Проверяются:

- количество observations и fields;
- названия columns;
- количество non-null values;
- тип данных каждого field;
- общее memory usage.

Особое внимание уделяется соответствию технического data type бизнес-семантике каждого field.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   checkout_session_id       80000 non-null  str  
 1   session_date              80000 non-null  str  
 2   city                      80000 non-null  str  
 3   delivery_options          80000 non-null  str  
 4   delivery_step_reached     80000 non-null  bool 
 5   selected_delivery_method  66667 non-null  str  
 6   payment_step_reached      80000 non-null  bool 
 7   order_completed           80000 non-null  bool 
dtypes: bool(3), str(5)
memory usage: 3.3 MB


### Missing Values Assessment

Первичная inspection показала, что `selected_delivery_method` содержит missing values, тогда как остальные fields заполнены полностью.

Проверяется количество и доля missing values по каждому field.

Наличие missing value не рассматривается автоматически как Data Quality issue: результат должен интерпретироваться с учётом Dataset Contract и семантики соответствующего field.

In [6]:
missing_values = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_rate_pct": df.isna().mean().mul(100).round(2),
    }
)

missing_values

,missing_count,missing_rate_pct
checkout_session_id,0,0.00
session_date,0,0.00
city,0,0.00
delivery_options,0,0.00
delivery_step_reached,0,0.00
selected_delivery_method,13333,16.67
payment_step_reached,0,0.00
order_completed,0,0.00


#### Interpretation

Missing values обнаружены только в `selected_delivery_method`:

- 13 333 observations;
- 16.67% анализируемых Checkout sessions.

Согласно Dataset Contract, отсутствие `selected_delivery_method` является допустимым состоянием и может означать, что в session отсутствует зафиксированный последний выбранный delivery method.

Поэтому эти observations не удаляются и не заполняются искусственным значением. Их поведение будет исследовано отдельно на последующих этапах анализа.

### Validation и преобразование `session_date`

Первичная inspection показала, что `session_date` загружен как `str`, хотя по Dataset Contract поле представляет календарную дату.

До изменения исходного DataFrame проверяется, могут ли все значения быть однозначно преобразованы в ожидаемый формат `YYYY-MM-DD`.

Некорректные значения при проверке преобразуются в `NaT`, что позволяет отдельно посчитать возможные parsing failures.

In [7]:
parsed_session_date = pd.to_datetime(
    df["session_date"],
    format="%Y-%m-%d",
    errors="coerce",
)

parsed_session_date.isna().sum()

np.int64(0)

Все значения `session_date` успешно прошли parsing validation. Нераспознанные даты отсутствуют, поэтому validated representation может безопасно заменить исходное строковое поле.

In [8]:
df["session_date"] = parsed_session_date

df["session_date"].dtype

dtype('<M8[us]')

### Проверка временного покрытия

Product Analytics заявил период extract с 2026-08-17 по 2026-09-13 — четыре полные недели.

Проверяются:

- минимальная дата;
- максимальная дата;
- количество уникальных календарных дат.

После этого отдельно проверяется отсутствие пропущенных дней внутри заявленного периода.

In [9]:
date_coverage = pd.DataFrame(
    {
        "metric": [
            "min_date",
            "max_date",
            "unique_dates",
        ],
        "value": [
            df["session_date"].min(),
            df["session_date"].max(),
            df["session_date"].nunique(),
        ],
    }
)

date_coverage

,metric,value
0,min_date,2026-08-17 00:00:00
1,max_date,2026-09-13 00:00:00
2,unique_dates,28


In [10]:
expected_dates = pd.date_range(
    start="2026-08-17",
    end="2026-09-13",
    freq="D",
)

missing_dates = expected_dates.difference(
    df["session_date"].unique()
)

missing_dates

DatetimeIndex([], dtype='datetime64[us]', freq='D')

#### Interpretation

Dataset охватывает период с 2026-08-17 по 2026-09-13 и содержит observations для всех 28 календарных дней.

Пропущенных календарных дат внутри заявленного периода не обнаружено.

Проверка подтверждает календарное покрытие extract, но сама по себе не доказывает полноту всех Checkout sessions внутри каждого дня.

### Проверка grain и uniqueness

Dataset Contract определяет grain как:

**1 row = 1 unique Checkout session.**

Для проверки этого утверждения сопоставляется общее количество observations с количеством уникальных `checkout_session_id` и отдельно проверяется наличие duplicates.

In [11]:
session_quality = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "unique_session_ids",
            "duplicate_session_ids",
        ],
        "value": [
            len(df),
            df["checkout_session_id"].nunique(),
            df["checkout_session_id"].duplicated().sum(),
        ],
    }
)

session_quality

,metric,value
0,row_count,80000
1,unique_session_ids,80000
2,duplicate_session_ids,0


#### Interpretation

Все 80 000 observations имеют уникальный `checkout_session_id`.

Duplicates не обнаружены, поэтому заявленный session-level grain подтверждается в рамках выполненной проверки.

### Domain Values Validation

Проверяются фактически присутствующие значения categorical fields:

- `city`;
- `delivery_options`;
- `selected_delivery_method`.

Цель проверки — выявить неожиданные или недопустимые domain values до использования этих fields для segmentation analysis.

Missing values в `selected_delivery_method` включаются в результат явно.

In [12]:
for column in [
    "city",
    "delivery_options",
    "selected_delivery_method",
]:
    print(f"{column}:")
    print(df[column].value_counts(dropna=False))
    print()

city:
city
Kyiv       30346
Kharkiv    20103
Dnipro     15096
Odesa      14455
Name: count, dtype: int64

delivery_options:
delivery_options
courier_only          46673
courier_and_pickup    33327
Name: count, dtype: int64

selected_delivery_method:
selected_delivery_method
courier    54343
NaN        13333
pickup     12324
Name: count, dtype: int64



### Logical Consistency Checks

Помимо schema и domain values, dataset проверяется на логическую согласованность связанных полей.

Проверяются следующие invariants:

1. Все observations относятся к sessions, достигшим Delivery step.
2. `order_completed = True` предполагает `payment_step_reached = True`.
3. `selected_delivery_method = pickup` невозможен при `delivery_options = courier_only`.
4. Выбранный delivery method должен входить в набор методов, доступных для соответствующей session.

In [13]:
consistency_checks = pd.DataFrame(
    {
        "check": [
            "delivery_step_not_reached",
            "order_completed_without_payment",
            "pickup_selected_when_courier_only",
        ],
        "violations": [
            (~df["delivery_step_reached"]).sum(),
            (
                df["order_completed"]
                & ~df["payment_step_reached"]
            ).sum(),
            (
                (df["delivery_options"] == "courier_only")
                & (df["selected_delivery_method"] == "pickup")
            ).sum(),
        ],
    }
)

consistency_checks

,check,violations
0,delivery_step_not_reached,0
1,order_completed_without_payment,0
2,pickup_selected_when_courier_only,0


### Data Quality Result

Initial Data Quality Assessment не выявил blocking issues для дальнейшего анализа.

Подтверждено:

- dataset содержит 80 000 observations и 80 000 уникальных `checkout_session_id`;
- duplicate Checkout sessions не обнаружены;
- период покрывает все 28 календарных дней с 2026-08-17 по 2026-09-13;
- `session_date` успешно преобразован в datetime без нераспознанных значений;
- обязательные поля не содержат missing values;
- 13 333 missing values в `selected_delivery_method` соответствуют заявленной семантике поля и сохраняются для дальнейшего анализа;
- значения `city`, `delivery_options` и `selected_delivery_method` соответствуют ожидаемым domains;
- проверенные logical consistency rules не имеют violations.

Dataset считается пригодным для текущего exploratory analysis с учётом ограничений, зафиксированных в Dataset Contract.

## Delivery → Payment Funnel Analysis

Первый этап business analysis оценивает переход Checkout sessions с Delivery step на Payment.

### Metric Definitions

**Delivery → Payment conversion**

Доля Checkout sessions, достигших Delivery step и впоследствии перешедших на Payment в рамках той же Checkout session.

**Delivery → Payment drop-off**

Доля Checkout sessions, достигших Delivery step, но не перешедших на Payment в рамках той же Checkout session.

`payment_step_reached` отражает переход на Payment и не означает успешную оплату или создание заказа.

In [14]:
delivery_sessions = len(df)
payment_sessions = df["payment_step_reached"].sum()
dropoff_sessions = (~df["payment_step_reached"]).sum()

delivery_sessions, payment_sessions, dropoff_sessions

(80000, np.int64(62342), np.int64(17658))

In [15]:
conversion_rate = payment_sessions / delivery_sessions
dropoff_rate = dropoff_sessions / delivery_sessions

funnel_metrics = pd.DataFrame(
    {
        "metric": [
            "delivery_sessions",
            "payment_sessions",
            "dropoff_sessions",
            "conversion_rate_pct",
            "dropoff_rate_pct",
        ],
        "value": [
            delivery_sessions,
            payment_sessions,
            dropoff_sessions,
            round(conversion_rate * 100, 2),
            round(dropoff_rate * 100, 2),
        ],
    }
)

funnel_metrics

,metric,value
0,delivery_sessions,80000.00
1,payment_sessions,62342.00
2,dropoff_sessions,17658.00
3,conversion_rate_pct,77.93
4,dropoff_rate_pct,22.07


In [16]:
round((conversion_rate + dropoff_rate) * 100, 10)

np.float64(100.0)

### Finding 1 — Overall Delivery → Payment Drop-off

В анализируемом extract:

- 80 000 Checkout sessions достигли Delivery step;
- 62 342 sessions (77.93%) перешли на Payment;
- 17 658 sessions (22.07%) не перешли на Payment.

Таким образом, измеренный Delivery → Payment drop-off для текущей выборки составляет **22.07%**.

Результат подтверждает наличие существенного наблюдаемого drop-off между Delivery и Payment в анализируемой population.

При этом показатель отличается от предварительной оценки Product Analytics (~18%), озвученной во время initial Discovery. Причина расхождения пока не установлена и требует уточнения metric definition, population, периода и применённых exclusions.

Текущий результат не устанавливает причину drop-off и не подтверждает, что ограниченный набор delivery options является его причиной.

### Metric Reconciliation

Предварительная оценка Product Analytics (~18%) и рассчитанный по extract Delivery → Payment drop-off (22.07%) используют разные metric definitions.

Основной Checkout dashboard применяет 24-hour session recovery rule: если customer возвращается и продолжает тот же Checkout в пределах установленного recovery window, такой случай не учитывается как окончательный abandonment на Delivery.

Текущий extract является session-level dataset и определяет переход на Payment только в рамках той же `checkout_session_id`. Cross-session recovery в dataset не объединяется.

Для текущего исследования используется **session-level Delivery → Payment drop-off = 22.07%**, поскольку дальнейший анализ `delivery_options` выполняется на той же единице анализа и в рамках одной согласованной population.

Показатель не должен напрямую сравниваться с dashboard abandonment rate без учёта различий в metric definition.

## Segmentation Analysis

### Delivery → Payment по `city`

Перед сравнением cohorts по `delivery_options` необходимо проверить, насколько Delivery → Payment behavior различается между городами.

Geography является потенциальным confounding factor, поскольку:

- доступность Pickup зависит от location;
- состав Checkout sessions может различаться между городами;
- baseline Delivery → Payment drop-off также может различаться географически.

Если распределение `delivery_options` и уровень drop-off одновременно зависят от `city`, агрегированное сравнение delivery-option cohorts может давать искажённое представление о наблюдаемой связи.

На этом этапе проверяются:

- количество Delivery sessions по каждому `city`;
- количество sessions, перешедших на Payment;
- количество sessions с Delivery → Payment drop-off;
- conversion rate;
- drop-off rate.

In [17]:
city_metrics = (
    df.groupby("city", as_index=False)
    .agg(
        delivery_sessions=("checkout_session_id", "count"),
        payment_sessions=("payment_step_reached", "sum"),
    )
)

city_metrics["dropoff_sessions"] = (
    city_metrics["delivery_sessions"]
    - city_metrics["payment_sessions"]
)

city_metrics["conversion_rate_pct"] = (
    city_metrics["payment_sessions"]
    / city_metrics["delivery_sessions"]
    * 100
).round(2)

city_metrics["dropoff_rate_pct"] = (
    city_metrics["dropoff_sessions"]
    / city_metrics["delivery_sessions"]
    * 100
).round(2)

city_metrics

,city,delivery_sessions,payment_sessions,dropoff_sessions,conversion_rate_pct,dropoff_rate_pct
0,Dnipro,15096,11583,3513,76.73,23.27
1,Kharkiv,20103,15379,4724,76.50,23.50
2,Kyiv,30346,24152,6194,79.59,20.41
3,Odesa,14455,11228,3227,77.68,22.32


### Finding 2 — Geographic Variation

Delivery → Payment drop-off различается между анализируемыми городами:

- Kharkiv — 23.50%;
- Dnipro — 23.27%;
- Odesa — 22.32%;
- Kyiv — 20.41%.

Разница между максимальным и минимальным наблюдаемым drop-off составляет **3.09 percentage points**.

Kyiv представляет крупнейшую часть анализируемой population и одновременно имеет наиболее низкий Delivery → Payment drop-off среди четырёх городов.

Результат показывает, что geography связана с наблюдаемым Checkout behavior и должна учитываться при последующем сравнении cohorts по `delivery_options`.

Сам по себе `city` не интерпретируется как причина различий в drop-off.

### Визуализация geographic variation

Для визуального сравнения Delivery → Payment drop-off между городами используется interactive bar chart.

Основной показатель — `dropoff_rate_pct`. Hover предоставляет дополнительный контекст по каждому geographic segment:

- точный Delivery → Payment drop-off;
- количество Delivery sessions;
- количество sessions с drop-off.

Y-axis начинается с нуля, чтобы визуально не преувеличивать различия между городами.

In [18]:
city_plot = city_metrics.sort_values(
    "dropoff_rate_pct",
    ascending=False,
)

fig = px.bar(
    city_plot,
    x="city",
    y="dropoff_rate_pct",
    color="city",
    text="dropoff_rate_pct",
    custom_data=[
        "delivery_sessions",
        "dropoff_sessions",
    ],
    title="Delivery → Payment Drop-off by City",
    labels={
        "city": "City",
        "dropoff_rate_pct": "Drop-off rate (%)",
    },
)

fig.update_traces(
    texttemplate="%{y:.2f}%",
    textposition="outside",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Delivery sessions: %{customdata[0]:,.0f}<br>"
        "Drop-off sessions: %{customdata[1]:,.0f}"
        "<extra></extra>"
    ),
)

fig.update_yaxes(
    range=[0, 27],
)

fig.update_layout(
    showlegend=False,
)

fig.show()

### Delivery → Payment по `delivery_options`

Исходная Product hypothesis предполагает, что ограниченный выбор delivery methods может быть связан с прекращением Checkout на Delivery step.

Для первичной проверки сравниваются две наблюдаемые cohorts:

- `courier_only` — sessions, в которых после первого успешного availability calculation был доступен только Courier;
- `courier_and_pickup` — sessions, в которых были доступны Courier и Pickup.

Для каждой cohort рассчитываются:

- количество Delivery sessions;
- количество sessions, перешедших на Payment;
- количество sessions с Delivery → Payment drop-off;
- conversion rate;
- drop-off rate.

Сравнение является observational и позволяет оценить association между `delivery_options` и Checkout behavior, но не устанавливает causal effect доступности Pickup.

In [19]:
delivery_options_metrics = (
    df.groupby("delivery_options", as_index=False)
    .agg(
        delivery_sessions=("checkout_session_id", "count"),
        payment_sessions=("payment_step_reached", "sum"),
    )
)

delivery_options_metrics["dropoff_sessions"] = (
    delivery_options_metrics["delivery_sessions"]
    - delivery_options_metrics["payment_sessions"]
)

delivery_options_metrics["conversion_rate_pct"] = (
    delivery_options_metrics["payment_sessions"]
    / delivery_options_metrics["delivery_sessions"]
    * 100
).round(2)

delivery_options_metrics["dropoff_rate_pct"] = (
    delivery_options_metrics["dropoff_sessions"]
    / delivery_options_metrics["delivery_sessions"]
    * 100
).round(2)

delivery_options_metrics

,delivery_options,delivery_sessions,payment_sessions,dropoff_sessions,conversion_rate_pct,dropoff_rate_pct
0,courier_and_pickup,33327,27506,5821,82.53,17.47
1,courier_only,46673,34836,11837,74.64,25.36


In [20]:
dropoff_by_option = (
    delivery_options_metrics
    .set_index("delivery_options")["dropoff_rate_pct"]
)

dropoff_difference_pp = (
    dropoff_by_option["courier_only"]
    - dropoff_by_option["courier_and_pickup"]
)

round(dropoff_difference_pp, 2)

np.float64(7.89)

### Контроль geographic composition

Aggregate comparison показывает разницу **7.89 percentage points** между `courier_only` и `courier_and_pickup` cohorts.

Однако geography уже продемонстрировала связь с Delivery → Payment drop-off и одновременно может влиять на доступность Pickup.

Поэтому aggregate association необходимо проверить внутри каждого `city`.

Если более высокий drop-off для `courier_only` сохраняется внутри всех или большинства geographic segments, это уменьшает вероятность того, что наблюдаемая aggregate difference объясняется только различиями geographic composition.

Такой stratified comparison по-прежнему остаётся observational и не устанавливает causal effect.

In [21]:
city_option_metrics = (
    df.groupby(
        ["city", "delivery_options"],
        as_index=False,
    )
    .agg(
        delivery_sessions=("checkout_session_id", "count"),
        payment_sessions=("payment_step_reached", "sum"),
    )
)

city_option_metrics["dropoff_sessions"] = (
    city_option_metrics["delivery_sessions"]
    - city_option_metrics["payment_sessions"]
)

city_option_metrics["dropoff_rate_pct"] = (
    city_option_metrics["dropoff_sessions"]
    / city_option_metrics["delivery_sessions"]
    * 100
).round(2)

city_option_metrics

,city,delivery_options,delivery_sessions,payment_sessions,dropoff_sessions,dropoff_rate_pct
0,Dnipro,courier_and_pickup,5428,4388,1040,19.16
1,Dnipro,courier_only,9668,7195,2473,25.58
2,Kharkiv,courier_and_pickup,7790,6387,1403,18.01
3,Kharkiv,courier_only,12313,8992,3321,26.97
4,Kyiv,courier_and_pickup,14790,12410,2380,16.09
5,Kyiv,courier_only,15556,11742,3814,24.52
6,Odesa,courier_and_pickup,5319,4321,998,18.76
7,Odesa,courier_only,9136,6907,2229,24.40


In [22]:
city_option_matrix = city_option_metrics.pivot(
    index="city",
    columns="delivery_options",
    values="dropoff_rate_pct",
)

city_option_matrix["difference_pp"] = (
    city_option_matrix["courier_only"]
    - city_option_matrix["courier_and_pickup"]
)

city_option_matrix.round(2)

delivery_options,courier_and_pickup,courier_only,difference_pp
city,,,
Dnipro,19.16,25.58,6.42
Kharkiv,18.01,26.97,8.96
Kyiv,16.09,24.52,8.43
Odesa,18.76,24.40,5.64


### Finding 3 — Association между `delivery_options` и Delivery → Payment Drop-off

В aggregate comparison наблюдается существенная разница Delivery → Payment drop-off между delivery-option cohorts:

- `courier_only` — 25.36%;
- `courier_and_pickup` — 17.47%;
- observed difference — **7.89 percentage points**.

После stratification по `city` направление association сохраняется во всех четырёх geographic segments:

- Dnipro — +6.42 pp для `courier_only`;
- Kharkiv — +8.96 pp;
- Kyiv — +8.43 pp;
- Odesa — +5.64 pp.

Таким образом, более высокий Delivery → Payment drop-off для `courier_only` не объясняется исключительно aggregate geographic composition анализируемых cohorts.

Результат усиливает исходную Product hypothesis о возможной связи между доступностью delivery options и Checkout behavior, но не доказывает causal effect Pickup.

Другие потенциальные confounding factors — включая характеристики заказа, customer composition, availability conditions и operational constraints — в текущем dataset не контролируются.

### Визуализация association по `city` и `delivery_options`

Grouped bar chart используется для сравнения Delivery → Payment drop-off между `courier_only` и `courier_and_pickup` внутри каждого geographic segment.

Сравнение внутри `city` позволяет визуально отделить observed delivery-option association от различий aggregate geographic composition.

Точные drop-off rates отображаются над bars; hover предоставляет размеры соответствующих cohorts и абсолютное количество drop-off sessions.

In [23]:
fig = px.bar(
    city_option_metrics,
    x="city",
    y="dropoff_rate_pct",
    color="delivery_options",
    barmode="group",
    text="dropoff_rate_pct",
    custom_data=[
        "delivery_sessions",
        "dropoff_sessions",
    ],
    title="Delivery → Payment Drop-off by City and Delivery Options",
    labels={
        "city": "City",
        "dropoff_rate_pct": "Drop-off rate (%)",
        "delivery_options": "Delivery options",
    },
)

fig.update_traces(
    texttemplate="%{y:.2f}%",
    textposition="outside",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Delivery sessions: %{customdata[0]:,.0f}<br>"
        "Drop-off sessions: %{customdata[1]:,.0f}"
        "<extra>%{fullData.name}</extra>"
    ),
)

fig.update_yaxes(
    range=[0, 31],
)

fig.show()

### Анализ отсутствующего `selected_delivery_method`

На этапе Data Quality Assessment было обнаружено 13 333 Checkout sessions (16.67%) без значения `selected_delivery_method`.

Согласно Dataset Contract, missing value является допустимым состоянием и не рассматривается автоматически как Data Quality issue.

На этом этапе проверяется, как отсутствие зафиксированного delivery method связано с:

- доступными `delivery_options`;
- переходом с Delivery на Payment.

Цель анализа — определить, представляет ли группа sessions без зафиксированного выбора отдельный behavioral signal, требующий дальнейшего Discovery.

`selected_delivery_method` содержит последний зафиксированный delivery method и не представляет полную историю взаимодействия customer с Delivery UI. Поэтому missing value сам по себе не устанавливает причину прекращения Checkout.

In [24]:
selection_by_options = (
    df.assign(
        selection_status=df["selected_delivery_method"]
        .notna()
        .map(
            {
                True: "method_selected",
                False: "no_method_selected",
            }
        )
    )
    .groupby(
        ["delivery_options", "selection_status"],
        as_index=False,
    )
    .agg(
        delivery_sessions=("checkout_session_id", "count"),
    )
)

selection_by_options["share_within_options_pct"] = (
    selection_by_options["delivery_sessions"]
    / selection_by_options.groupby("delivery_options")[
        "delivery_sessions"
    ].transform("sum")
    * 100
).round(2)

selection_by_options

,delivery_options,selection_status,delivery_sessions,share_within_options_pct
0,courier_and_pickup,method_selected,28320,84.98
1,courier_and_pickup,no_method_selected,5007,15.02
2,courier_only,method_selected,38347,82.16
3,courier_only,no_method_selected,8326,17.84


In [25]:
selection_payment_metrics = (
    df.assign(
        selection_status=df["selected_delivery_method"]
        .notna()
        .map(
            {
                True: "method_selected",
                False: "no_method_selected",
            }
        )
    )
    .groupby(
        ["delivery_options", "selection_status"],
        as_index=False,
    )
    .agg(
        delivery_sessions=("checkout_session_id", "count"),
        payment_sessions=("payment_step_reached", "sum"),
    )
)

selection_payment_metrics["dropoff_sessions"] = (
    selection_payment_metrics["delivery_sessions"]
    - selection_payment_metrics["payment_sessions"]
)

selection_payment_metrics["dropoff_rate_pct"] = (
    selection_payment_metrics["dropoff_sessions"]
    / selection_payment_metrics["delivery_sessions"]
    * 100
).round(2)

selection_payment_metrics

,delivery_options,selection_status,delivery_sessions,payment_sessions,dropoff_sessions,dropoff_rate_pct
0,courier_and_pickup,method_selected,28320,24140,4180,14.76
1,courier_and_pickup,no_method_selected,5007,3366,1641,32.77
2,courier_only,method_selected,38347,29839,8508,22.19
3,courier_only,no_method_selected,8326,4997,3329,39.98


### Finding 4 — Delivery Method Selection как behavioral signal

Отсутствие зафиксированного `selected_delivery_method` связано с существенно более высоким Delivery → Payment drop-off в обеих availability cohorts.

Для `courier_and_pickup`:

- `method_selected` — 14.76% drop-off;
- `no_method_selected` — 32.77%;
- observed difference — **18.01 percentage points**.

Для `courier_only`:

- `method_selected` — 22.19% drop-off;
- `no_method_selected` — 39.98%;
- observed difference — **17.79 percentage points**.

При этом отсутствие зафиксированного method встречается несколько чаще в `courier_only` cohort: 17.84% против 15.02% для `courier_and_pickup`.

Даже среди sessions с зафиксированным delivery method более высокий drop-off сохраняется для `courier_only`: 22.19% против 14.76%.

Результат показывает, что отсутствие зафиксированного delivery method является сильным behavioral signal, связанным с прекращением Checkout, но не устанавливает причину такого поведения.

Текущий dataset не позволяет определить, означает ли missing `selected_delivery_method` отсутствие подходящего варианта, отсутствие взаимодействия с selector, техническую особенность event tracking или иной customer behavior.

### Interpretation и аналитическое ограничение

Уточнение с Product Analytics подтвердило, что `selected_delivery_method` отражает наличие подтверждённого delivery method в последнем зафиксированном Delivery state.

Для Pickup значение появляется после выбора конкретной pickup location. Для Courier подтверждение method зависит от состояния Checkout и в отдельных сценариях может требовать дополнительного действия или повторного выбора после recalculation.

Поэтому missing `selected_delivery_method` представляет отсутствие подтверждённого delivery method в наблюдаемом состоянии, но не идентифицирует единственную причину этого состояния.

Текущая event instrumentation позволяет наблюдать отдельные взаимодействия, включая открытие pickup selector, выбор pickup location и смену delivery method, однако не позволяет надёжно различить все возможные причины отсутствия выбора.

В частности, текущий dataset не позволяет установить, означает ли `no_method_selected`:

- отсутствие подходящего delivery option;
- просмотр доступных вариантов без выбора;
- отсутствие взаимодействия с selector;
- неподтверждённый Courier после availability recalculation;
- иной UI state или tracking limitation.

Поэтому повышенный drop-off для `no_method_selected` рассматривается как behavioral signal для дальнейшего Discovery, а не как доказательство конкретной причины abandonment.